# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. 

**Dataset source:** the dataset is described by a Croissant schema URL and organized into multiple record sets and fields, allowing robust and interoperable data access.

> **Note:**
All dataset entities (record sets, fields, columns) are referenced _by their `@id`_ fields to ensure clarity and reproducibility.

### Dataset Source
- Croissant JSON-LD: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- License: [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata (does not load all records yet)
dataset = mlc.Dataset(croissant_url)

# Show dataset overview
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Authors: {dataset.metadata.author}")
print(f"Keywords: {dataset.metadata.keywords}")


## 2. Data Overview
Review available record sets, their field `@id`s, and the schema structure.

In [ ]:
# The Croissant API provides metadata as a graph of entities. Let's inspect record sets:

# List all record sets and their IDs
record_set_objs = list(dataset.metadata.record_sets)
print(f'Found {len(record_set_objs)} record set(s)')
for rs in record_set_objs:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', None)}")
    # List the fields of this record set
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id} (name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)})")


## 3. Data Extraction
Load data from a specific record set (referenced by its `@id`) into a pandas DataFrame for analysis. Use the `@id`s found above.

In [ ]:
# Select all record set @ids for extraction. Example: there may be one main data table.
record_set_ids = [rs.id for rs in record_set_objs]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        # Many Croissant tables are wide; show a sample
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("No records found or record set may describe non-tabular data.")


### Which record set contains the main experiment table?
Based on the dataset's description, the primary DataFrame will typically have columns with protein information (e.g. accession, description, coverage, peptide counts, MW, pI, and abundances). Refer to the previous cell for the correct record set `@id`.

> **For purposes of this notebook, we will use the first record set with tabular data as the main analysis table.**

In [ ]:
# Select main record set for demonstration (the first containing tabular data)
main_record_set_id = None
for rid, df in dataframes.items():
    if df.shape[0] > 0:  # Any records present
        main_record_set_id = rid
        main_df = df
        break

if main_record_set_id is None:
    raise RuntimeError("No suitable tabular record set found in this dataset.")
print(f"Main analysis RecordSet @id: {main_record_set_id}")
print(main_df.head())


## 4. Exploratory Data Analysis (EDA)
Typical EDA steps: filter records by abundance, normalize abundance, group by protein type/description, etc.

> _Choose appropriate numeric and grouping fields using their `@id` values as shown in the schema overview._

In [ ]:
# Identify a numeric field (by @id) for EDA, e.g. 'cr:coverage' (protein coverage percentage) or 'cr:mw' (molecular weight)

# Try-known field names. If you have the schema handy, replace the variables as needed.
possible_numeric_fields = [c for c in main_df.columns if ('coverage' in c.lower() or 'abundance' in c.lower() or 'mw' in c.lower() or 'count' in c.lower() or 'cr:' in c)]
print('Possible numeric fields for analysis:', possible_numeric_fields)

# We'll pick the first one for demonstration; you can adjust this for your actual field of interest.
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # fallback: choose first column
    numeric_field_id = main_df.columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Filter records with numeric_field > threshold (set a threshold value)
threshold = main_df[numeric_field_id].astype(float).mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
filtered_df = main_df.copy()
try:
    filtered_df = filtered_df[filtered_df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered rows with {numeric_field_id} > {threshold:.2f} : {filtered_df.shape[0]} records")
    print(filtered_df.head())
except Exception as e:
    print(f"Could not filter on field {numeric_field_id}. Error: {e}")

# Normalize the numeric field (z-score within filtered records)
from numpy import nanmean, nanstd
if filtered_df.shape[0] > 0:
    vals = filtered_df[numeric_field_id].astype(float)
    filtered_df[f"{numeric_field_id}_normalized"] = (vals - nanmean(vals)) / nanstd(vals)
    print(f"Normalized {numeric_field_id} (z-score):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt grouping by a categorical field (e.g., description or type)
possible_group_fields = [c for c in main_df.columns if 'desc' in c.lower() or 'accession' in c.lower() or 'category' in c.lower()]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    group_field = None
    print("No suitable grouping field identified.")


## 5. Visualization
Let's plot the distribution of the selected numeric field, and—if available—a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id].astype(float), kde=True, bins=20)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id].astype(float))
    plt.xticks(rotation=45)
    plt.title(f'{numeric_field_id} by {group_field}')
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Using the Croissant FAIR^2 schema and the `mlcroissant` library, we've programmatically:  
  - Discovered dataset schema via `@id` references  
  - Loaded tabular data into pandas DataFrames  
  - Performed basic filtering, normalization, and grouping
  - Plotted basic distributions for quantitative exploration

**Further exploration:** use field `@id`s to repeat analyses for other experimental measures, link to documentation for full metadata, or join with other Croissant datasets. For scientific or clinical investigations, always consult the full [dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and license terms.